# Finetune a YOLO golf-bunker detector

Trains a YOLO11 model on the bunker dataset produced by `create_bunker_dataset.ipynb`, then publishes the trained weights to the HuggingFace Model Hub.

**Recommended environment:** Google Colab with a T4 GPU (free tier). Expected wall-clock for 10 epochs on ~1,700 train images at imgsz=512 with `yolo11m`: roughly 30–60 minutes.

## Dependencies

In [ ]:
%pip install --quiet ultralytics

## Download dataset from HuggingFace

You need to set the `HF_TOKEN` Colab secret:

- Create an account: https://huggingface.co/join
- Follow this guide about [`User Access Tokens`](https://huggingface.co/docs/hub/security-tokens)

Set `DATASET` to the dataset repo created by `create_bunker_dataset.ipynb`.

In [ ]:
from huggingface_hub import hf_hub_download

In [ ]:
DATASET = "JohnieWalkerLV/osm-golf-bunkers"

In [ ]:
hf_hub_download(
    DATASET, filename="train.zip", repo_type="dataset", local_dir="datasets"
)

In [ ]:
hf_hub_download(DATASET, filename="val.zip", repo_type="dataset", local_dir="datasets")

In [ ]:
hf_hub_download(
    DATASET, filename="yolo_dataset.yaml", repo_type="dataset", local_dir="datasets"
)

In [ ]:
!unzip -q -o datasets/train.zip

In [ ]:
!unzip -q -o datasets/val.zip

# Finetune model

In [ ]:
from ultralytics import YOLO

Check the [available models](https://docs.ultralytics.com/tasks/detect/#models). `yolo11m` is a reasonable default — large enough to learn small features like bunkers, small enough to train on free Colab GPU.

In [ ]:
MODEL = "yolo11m.pt"

In [ ]:
yolo = YOLO(MODEL)

Hyperparameter notes:

- `imgsz=512`: matches the tile size produced by the dataset notebook.
- `flipud=0.5`: vertical flips are safe for satellite imagery (no "up" axis).
- `scale=0.0`, `translate=0.0`: disabled because absolute pixel size of bunkers is meaningful at a fixed zoom level.
- `lr0=0.01` + `cos_lr=True` + `AdamW`: same recipe as the swimming-pool model.
- `patience=3`: stop early if validation stalls. With clean labels expect convergence around epoch 5–8.

In [ ]:
# Rewrite the YAML to use an absolute dataset path. Ultralytics' relative
# path resolution differs from the YAML's directory in some versions, so
# pinning to the absolute /content/datasets is the safest portable fix.
import yaml
from pathlib import Path

cfg_path = Path("datasets/yolo_dataset.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(Path("datasets").resolve())
cfg_path.write_text(yaml.dump(cfg))
print(cfg)

In [ ]:
yolo.train(
    data="datasets/yolo_dataset.yaml",
    patience=3,
    imgsz=512,
    scale=0.0,
    flipud=0.5,
    translate=0.0,
    cos_lr=True,
    exist_ok=True,
    optimizer="AdamW",
    lr0=0.01,
    epochs=10,
)

# Check results

Inspect predictions on the validation set. Boxes should land on actual sand bunkers, not greens, fairways, or paths.

In [ ]:
from PIL import Image

In [ ]:
Image.open("/content/runs/detect/train/val_batch0_pred.jpg")

If validation looks poor:
- Check `/content/runs/detect/train/results.png` for loss curves.
- Try a larger model (`yolo11l.pt` or `yolo11x.pt`) — needs more VRAM.
- Consider expanding the training set by raising `N_TRAIN` in the dataset notebook.
- If the model is finding bunker-like features that *aren't* tagged in OSM, this is the missing-label problem and is expected — those are real bunkers OSM hasn't recorded yet (which is exactly the point of the tool).

# Upload model

Set `USER` to your HuggingFace username. The model repo `{USER}/osm-bunker-detector` will be created if it doesn't exist.

In [ ]:
USER = "JohnieWalkerLV"
REPO = "osm-bunker-detector"

In [ ]:
import yaml

with open("datasets/yolo_dataset.yaml") as f:
    CLASS_NAME = yaml.safe_load(f)["names"][0]

In [ ]:
from pathlib import Path

Path("README.md").write_text(
    f"""
---
datasets:
- {DATASET}
base_model: Ultralytics/YOLO11
library_name: ultralytics
pipeline_tag: object-detection
license: apache-2.0
---

# {REPO}

Detect golf {CLASS_NAME}s (sand traps) in satellite imagery.

Created with [osm-ai-helper](https://github.com/mozilla-ai/osm-ai-helper).

## Training data

Trained on [{DATASET}](https://huggingface.co/datasets/{DATASET}), a small but high-quality dataset built from the best-mapped golf courses in OpenStreetMap.

## Use

```python
from huggingface_hub import hf_hub_download
from osm_ai_helper.run_inference import run_inference

model_file = hf_hub_download("{USER}/{REPO}", filename="model.pt", repo_type="model")
run_inference(
    yolo_model_file=model_file,
    output_dir="inference_out",
    lat_lon=(33.5021, -82.0229),  # Augusta National
    margin=4,
    selector="golf=bunker",
    zoom=19,
)
```
"""
)

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

In [ ]:
api = HfApi()

In [ ]:
try:
    api.create_repo(f"{USER}/{REPO}", token=userdata.get("HF_TOKEN"), repo_type="model")
except Exception:
    pass

In [ ]:
api.upload_file(
    token=userdata.get("HF_TOKEN"),
    path_or_fileobj="/content/runs/detect/train/weights/best.pt",
    path_in_repo="model.pt",
    repo_id=f"{USER}/{REPO}",
    repo_type="model",
)

In [ ]:
api.upload_file(
    token=userdata.get("HF_TOKEN"),
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=f"{USER}/{REPO}",
    repo_type="model",
)

In [ ]:
api.upload_file(
    token=userdata.get("HF_TOKEN"),
    path_or_fileobj="/content/runs/detect/train/val_batch0_pred.jpg",
    path_in_repo="val_batch0_pred.jpg",
    repo_id=f"{USER}/{REPO}",
    repo_type="model",
)